<a href="https://colab.research.google.com/github/SarahkhIT/DataEngProject/blob/main/notebooks/00_setup_lineage.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ResearchAtlas— Capstone

**Team:** Basma (Ingestion) · Rehaf (Lakehouse & Quality) · Sarah (RAG)

**Program:** Modern Data Engineering for AI Systems — SDAIA Academy, DAICO

**Trainer:** Mohammed Albeladi | **Session dates:** 19/july - 23/july

## Contents
1. Setup — OpenLineage lineage wrapper (shared by every stage)
2. Ingestion — Kafka & Pydantic
3. Delta Lakehouse — Bronze / Silver / Gold
4. Orchestration — Airflow DAG
5. RAG Pipeline — Retrieval & Generation


# 1. Setup — Shared OpenLineage Wrapper

Every stage in this notebook (ingestion, lakehouse, quality gate, RAG) emits real START/COMPLETE/FAIL events through this one wrapper, into one log file — this is the evidence for the Quality Gate + Lineage rubric line.

In [ ]:
!pip install -q kafka-python deltalake pyarrow great_expectations openlineage-python chromadb sentence-transformers rank-bm25 numpy groq apache-airflow apache-airflow-providers-standard

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 614.1/614.1 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.6/49.6 MB 22.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 72.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.3/114.3 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 46.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.4/6.4 MB 58.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 595.9/595.9 kB 29.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.3/161.3 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.9/43.9 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 54.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2

In [ ]:
from openlineage.client import OpenLineageClient
from openlineage.client.transport.file import FileConfig, FileTransport
from openlineage.client.event_v2 import RunEvent, RunState, Run, Job
from openlineage.client.uuid import generate_new_uuid
from datetime import datetime, UTC
import os

os.makedirs("lineage_events", exist_ok=True)
_transport = FileTransport(FileConfig(log_file_path="lineage_events/openlineage_run.log"))
_client = OpenLineageClient(transport=_transport)
_PRODUCER = "https://github.com/SarahkhIT/DataEngProject"  # update with your real repo URL

def run_with_lineage(job_name, fn, *args, **kwargs):
    """Wraps any pipeline stage function with real OpenLineage START/COMPLETE/FAIL events."""
    run_id = str(generate_new_uuid())
    job = Job(namespace="arxiv_capstone", name=job_name)
    run = Run(runId=run_id)

    _client.emit(RunEvent(eventType=RunState.START, eventTime=datetime.now(UTC).isoformat(), run=run, job=job, producer=_PRODUCER))
    print(f"[OpenLineage] START — {job_name}")

    try:
        result = fn(*args, **kwargs)
    except Exception as e:
        _client.emit(RunEvent(eventType=RunState.FAIL, eventTime=datetime.now(UTC).isoformat(), run=run, job=job, producer=_PRODUCER))
        print(f"[OpenLineage] FAIL — {job_name}: {e}")
        raise

    _client.emit(RunEvent(eventType=RunState.COMPLETE, eventTime=datetime.now(UTC).isoformat(), run=run, job=job, producer=_PRODUCER))
    print(f"[OpenLineage] COMPLETE — {job_name}")
    return result

print("✅ Lineage wrapper ready.")

2026-07-22T22:50:16.892175Z [info     ] OpenLineageClient will use `file` transport [openlineage.client.client] loc=client.py:136
✅ Lineage wrapper ready.
